# Recommendation 05 — Pipeline LLM/MCP v2

Il riferimento principale è **B, k=10**. I film senza giudizio danno zero guadagno osservato e non sono classificati come dislike. Le curve a k diversi hanno target B diversi. Il test già ispezionato rimane esplorativo. Ogni report indica modalità e sottoinsieme; le repliche sono mediate per utente prima del bootstrap.

Le celle sono intenzionalmente disattivate: attivare soltanto la fase desiderata. Nessun risultato precedente è sovrascritto. Vedere `docs/experiments/recommendation-coldstart-v2.md`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p/'configs/recommendation_coldstart_v2.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Aprire il notebook dalla repository o da notebooks/')
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
from movie_rag.recommendation_benchmark import BenchmarkConfig, Protocol
CONFIG = BenchmarkConfig.load(ROOT)
PROTOCOL_ID = CONFIG.values['protocol_id']
print('Protocollo:', PROTOCOL_ID, '| configurazione:', CONFIG.fingerprint[:12])


Profilo dedicato: ricerca esatta, 30 candidati, nessun reranker e nessun filtro rigido inferito. I film già rivelati sono rimossi prima della top 10. La metrica primaria usa il ranking MCP; la risposta Answer e le sue raccomandazioni restano nella traccia. Risultati vuoti validi restano nel confronto; errori infrastrutturali interrompono il run e richiedono ripresa. Preparare prima le baseline della stessa modalità e completare full_validation prima di test.

In [ ]:
RUN_MODE = 'pilot_validation'  # pilot_validation, full_validation, test
from movie_rag.recommendation_experiments import run_pipeline
RUN_LIVE = False
MAX_NEW_CALLS = CONFIG.values['llm']['max_new_calls']
if RUN_LIVE:
    protocol = Protocol.load(CONFIG)
    report = await run_pipeline(protocol, RUN_MODE, max_new_calls=MAX_NEW_CALLS)
    display(report['macro'].query("scheme == 'B' and k == 10"))
    display(report['paired'].query("scheme == 'B' and k == 10"))
    print('Report:', report['directory'])
else:
    print('Pipeline LLM/MCP/Qdrant disattivata. Budget workflow:', MAX_NEW_CALLS)
